# Lab Contract Reimbursement Analysis: Percent of Charges vs. Fee Schedule

   This project evaluates the financial impact of a proposed change to a rural hospital’s managed care contract with a major payer. Under the current agreement, outpatient services are reimbursed based on a percentage of billed charges. During contract renewal negotiations, the payer proposed moving outpatient laboratory services to a fixed fee schedule. The hospital requested an analysis of two years of outpatient lab activity to compare the current and proposed reimbursement methods and identify the tests driving the financial variance.
To protect confidential patient information and contract terms, this portfolio recreation uses fully synthetic visit, laboratory charge-master, and patient-charge data. The proposed payer fee schedule was also simulated using CMS Clinical Laboratory Fee Schedule rates increased by 25%, or 125% of the CMS rate. I performed the analysis with DuckDB SQL in a Jupyter Notebook.
This project is based on a real-world analysis I previously completed, but all data, reimbursement rates, and results presented here are synthetic and illustrative.

In [ ]:
**Tools:** Jupyter Notebook, DuckDB SQL, Python (DuckDB package), Tableau;  <br>**Data Source:** Synthetic data created for portfolio demonstration; no protected health information or actual contract materials included.

## Project Objectives

 - Quantify outpatient laboratory volume and billed charges from January 1, 2024 through December 31, 2025
 - Estimate reimbursement under the current percent-of-charges methodology
 - Model reimbursement under the proposed laboratory fee schedule
 - Compare the two methodologies to calculate the estimated dollar and percentage variance
 - Identify the laboratory tests contributing most to the projected gains or losses

In [1]:
import duckdb

## Source Files and Data Limitations

   The project uses synthetic CSV files stored in the 'Lab Contract Review' directory. The data was created solely to demonstrate data validation, relational data modeling, SQL analysis, and data visualization techniques. It does not represent real patients, encounters, or healthcare activity.

## Source Data Inventory

In [2]:
duckdb.sql("""
SELECT 'lab_charge_master' AS dataset,
COUNT(*) AS row_count
FROM 'Lab Contract Review/lab_charge_master.csv'

UNION ALL

SELECT 'patient_charges',
COUNT(*) AS row_count
FROM 'Lab Contract Review/patient_charges.csv'

UNION ALL

SELECT 'patient_encounters',
COUNT(*) AS row_count
FROM 'Lab Contract Review/patient_encounters.csv'

UNION ALL

SELECT 'lab_fee_schedule',
COUNT(*) AS row_count
FROM 'Lab Contract Review/lab_fee_schedule.csv';
""")

┌────────────────────┬───────────┐
│      dataset       │ row_count │
│      varchar       │   int64   │
├────────────────────┼───────────┤
│ lab_charge_master  │        81 │
│ patient_charges    │      1450 │
│ patient_encounters │      1200 │
│ lab_fee_schedule   │        81 │
└────────────────────┴───────────┘

The following queries display a small sample from each source table to show its structure and representative values before validation.

In [3]:
duckdb.sql("""
SELECT *
FROM 'Lab Contract Review/lab_charge_master.csv'
ORDER BY description
LIMIT 10;
""")

┌───────────┬───────────────────────────────────┬─────────┬───────┬────────┐
│ chargeNum │            description            │ revCode │  CPT  │ price  │
│  varchar  │              varchar              │ varchar │ int64 │ double │
├───────────┼───────────────────────────────────┼─────────┼───────┼────────┤
│ LAB100052 │ ABO Blood Typing                  │ 0305    │ 86900 │   88.0 │
│ LAB100036 │ Alanine Aminotransferase          │ 0301    │ 84460 │   59.0 │
│ LAB100007 │ Albumin                           │ 0301    │ 82040 │   62.0 │
│ LAB100030 │ Alkaline Phosphatase              │ 0301    │ 84075 │   61.0 │
│ LAB100009 │ Amylase                           │ 0301    │ 82150 │  104.0 │
│ LAB100068 │ Antimicrobial Susceptibility, MIC │ 0306    │ 87186 │  167.0 │
│ LAB100054 │ Antinuclear Antibody Screen       │ 0302    │ 86038 │  174.0 │
│ LAB100028 │ B-Type Natriuretic Peptide        │ 0301    │ 83880 │  319.0 │
│ LAB100066 │ Bacterial Culture, Other Source   │ 0306    │ 87070 │  139.0 │

In [4]:
duckdb.sql("""
SELECT *
FROM 'Lab Contract Review/patient_charges.csv'
ORDER BY svcDate
LIMIT 10;
""")

┌────────┬────────────┬───────────┬────────────┬───────┐
│ rowNum │  acctNum   │ chargeNum │  svcDate   │  qty  │
│ int64  │  varchar   │  varchar  │    date    │ int64 │
├────────┼────────────┼───────────┼────────────┼───────┤
│      1 │ ACCT000001 │ LAB100081 │ 2024-01-01 │     1 │
│      2 │ ACCT000002 │ LAB100077 │ 2024-01-01 │     1 │
│      3 │ ACCT000003 │ LAB100077 │ 2024-01-02 │     1 │
│      4 │ ACCT000004 │ LAB100048 │ 2024-01-03 │     1 │
│      5 │ ACCT000005 │ LAB100021 │ 2024-01-04 │     1 │
│      6 │ ACCT000006 │ LAB100018 │ 2024-01-04 │     1 │
│      7 │ ACCT000007 │ LAB100021 │ 2024-01-05 │     1 │
│      8 │ ACCT000008 │ LAB100077 │ 2024-01-05 │     1 │
│      9 │ ACCT000009 │ LAB100001 │ 2024-01-06 │     1 │
│     10 │ ACCT000009 │ LAB100031 │ 2024-01-06 │     1 │
└────────┴────────────┴───────────┴────────────┴───────┘
  10 rows                                    5 columns

In [5]:
duckdb.sql("""
SELECT *
FROM 'Lab Contract Review/patient_encounters.csv'
ORDER BY acctNum
LIMIT 10;
""")

┌────────────┬────────────┐
│  acctNum   │ admitDate  │
│  varchar   │    date    │
├────────────┼────────────┤
│ ACCT000001 │ 2024-01-01 │
│ ACCT000002 │ 2024-01-01 │
│ ACCT000003 │ 2024-01-02 │
│ ACCT000004 │ 2024-01-03 │
│ ACCT000005 │ 2024-01-04 │
│ ACCT000006 │ 2024-01-04 │
│ ACCT000007 │ 2024-01-05 │
│ ACCT000008 │ 2024-01-05 │
│ ACCT000009 │ 2024-01-06 │
│ ACCT000010 │ 2024-01-07 │
└────────────┴────────────┘
  10 rows       2 columns

In [6]:
duckdb.sql("""
SELECT *
FROM 'Lab Contract Review/lab_fee_schedule.csv'
ORDER BY CPT
LIMIT 10;
""")

┌───────┬───────────────────────────────┬────────┐
│  CPT  │          description          │  fee   │
│ int64 │            varchar            │ double │
├───────┼───────────────────────────────┼────────┤
│ 80048 │ Basic Metabolic Panel         │  10.58 │
│ 80051 │ Electrolyte Panel             │   8.76 │
│ 80053 │ Comprehensive Metabolic Panel │   13.2 │
│ 80061 │ Lipid Panel                   │  16.74 │
│ 80069 │ Renal Function Panel          │  10.85 │
│ 80076 │ Hepatic Function Panel        │  10.21 │
│ 80162 │ Digoxin Level                 │   16.6 │
│ 80202 │ Vancomycin Level              │  16.93 │
│ 80307 │ Presumptive Drug Screen       │  77.68 │
│ 81001 │ Urinalysis with Microscopy    │   3.96 │
└───────┴───────────────────────────────┴────────┘
  10 rows                              3 columns

## Source Data Validation

### 'lab_charge_master' Validation

The following query verifies that each charge number uniquely identifies one record.

In [7]:
duckdb.sql("""
SELECT
   COUNT (*) AS row_count,
   COUNT (DISTINCT chargeNum) as distinct_key_count
FROM 'Lab Contract Review/lab_charge_master.csv';
""")

┌───────────┬────────────────────┐
│ row_count │ distinct_key_count │
│   int64   │       int64        │
├───────────┼────────────────────┤
│        81 │                 81 │
└───────────┴────────────────────┘

The following query identifies any charge numbers that appear more than once. Because 'chargeNum' is the primary key for the charge master, each value should occur only once. A successful validation returns no rows.

In [8]:
duckdb.sql("""
SELECT
   chargeNum,
   COUNT(*) AS occurrences
FROM 'Lab Contract Review/lab_charge_master.csv'
GROUP BY chargeNum
HAVING COUNT(*) > 1;
""")

┌───────────┬─────────────┐
│ chargeNum │ occurrences │
│  varchar  │    int64    │
└───────────┴─────────────┘
          0 rows         

The following query checks required fields for missing values.

In [9]:
duckdb.sql("""
SELECT
    COUNT(*) FILTER (WHERE chargeNum IS NULL) AS "Missing Charge Numbers"
FROM 'Lab Contract Review/lab_charge_master.csv';
""")

┌────────────────────────┐
│ Missing Charge Numbers │
│         int64          │
├────────────────────────┤
│                      0 │
└────────────────────────┘

### 'patient_charges' Validation

The following query verifies that the sequential rowNum field uniquely identifies every charge record.

In [10]:
duckdb.sql("""
SELECT
   COUNT (*) AS row_count,
   COUNT (DISTINCT rowNum) as distinct_key_count
FROM 'Lab Contract Review/patient_charges.csv';
""")

┌───────────┬────────────────────┐
│ row_count │ distinct_key_count │
│   int64   │       int64        │
├───────────┼────────────────────┤
│      1450 │               1450 │
└───────────┴────────────────────┘

Charge corrections may produce multiple records for the same account, charge number, and service date because an incorrect charge is retained with an offsetting negative quantity. The following query excludes fully reversed charges and identifies repeated charge combinations that retain a positive net quantity for further review.

In [11]:
duckdb.sql("""
SELECT
    acctNum,
    chargeNum,
    svcDate,
    COUNT(*) AS transaction_count,
    SUM(qty) AS net_quantity
FROM 'Lab Contract Review/patient_charges.csv'
GROUP BY
    acctNum,
    chargeNum,
    svcDate
HAVING COUNT(*) > 1
   AND SUM(qty) > 0
ORDER BY
    acctNum,
    svcDate,
    chargeNum;
""")

┌─────────┬───────────┬─────────┬───────────────────┬──────────────┐
│ acctNum │ chargeNum │ svcDate │ transaction_count │ net_quantity │
│ varchar │  varchar  │  date   │       int64       │    int128    │
└─────────┴───────────┴─────────┴───────────────────┴──────────────┘
                               0 rows                             

### 'patient_encounters' Validation

The following query verifies that each account number identifies one unique record.

In [12]:
duckdb.sql("""
SELECT
   COUNT (*) AS row_count,
   COUNT (DISTINCT acctNum) as distinct_key_count
FROM 'Lab Contract Review/patient_encounters.csv';
""")

┌───────────┬────────────────────┐
│ row_count │ distinct_key_count │
│   int64   │       int64        │
├───────────┼────────────────────┤
│      1200 │               1200 │
└───────────┴────────────────────┘

The following query checks required fields for missing values.

In [13]:
duckdb.sql("""
SELECT
    COUNT(*) FILTER (WHERE acctNum IS NULL) AS "Missing Account Numbers"
FROM 'Lab Contract Review/patient_encounters.csv';
""")

┌─────────────────────────┐
│ Missing Account Numbers │
│          int64          │
├─────────────────────────┤
│                       0 │
└─────────────────────────┘

### 'lab_fee_schedule' Validation

The following query verifies that each CPT code identifies one unique record.

In [14]:
duckdb.sql("""
SELECT
   COUNT (*) AS row_count,
   COUNT (DISTINCT CPT) as distinct_key_count
FROM 'Lab Contract Review/lab_fee_schedule.csv';
""")

┌───────────┬────────────────────┐
│ row_count │ distinct_key_count │
│   int64   │       int64        │
├───────────┼────────────────────┤
│        81 │                 81 │
└───────────┴────────────────────┘

The following query checks required fields for missing values.

In [15]:
duckdb.sql("""
SELECT
    COUNT(*) FILTER (WHERE CPT IS NULL) AS "Missing CPT Codes"
FROM 'Lab Contract Review/lab_fee_schedule.csv';
""")

┌───────────────────┐
│ Missing CPT Codes │
│       int64       │
├───────────────────┤
│                 0 │
└───────────────────┘

The following query identifies any CPT codes that appear more than once. Because `CPT` is the primary key for the lab fee schedule, each value should occur only once. A successful validation returns no rows.

In [16]:
duckdb.sql("""
SELECT
    CPT,
    COUNT(*) AS occurrences
FROM 'Lab Contract Review/lab_fee_schedule.csv'
GROUP BY CPT
HAVING COUNT(*) > 1;
""")

┌───────┬─────────────┐
│  CPT  │ occurrences │
│ int64 │    int64    │
└───────┴─────────────┘
        0 rows       

### Cross-File Validation

The following query joins the four validated source tables and displays all available columns to verify the structure of the combined records. The `patient_charges` table supplies the transaction-level activity, while `patient_encounters`, `lab_charge_master`, and `lab_fee_schedule` add the related encounter details, charge descriptions, current prices, and proposed reimbursement amounts. This full-column result is used to validate the joins before selecting the fields needed for the account-level analysis dataset.

In [17]:
duckdb.sql("""
SELECT
   *
FROM 'Lab Contract Review/patient_charges.csv' as c
JOIN 'Lab Contract Review/lab_charge_master.csv' as m
   ON c.chargeNum = m.chargeNum
JOIN 'Lab Contract Review/patient_encounters.csv' as v
   ON c.acctNum = v.acctNum
JOIN 'Lab Contract Review/lab_fee_schedule.csv' as f
   ON m.CPT = f.CPT
""")

┌────────┬────────────┬───────────┬────────────┬───────┬───────────┬────────────────────────────────────────┬─────────┬───────┬────────┬────────────┬────────────┬───────┬────────────────────────────────────────┬────────┐
│ rowNum │  acctNum   │ chargeNum │  svcDate   │  qty  │ chargeNum │              description               │ revCode │  CPT  │ price  │  acctNum   │ admitDate  │  CPT  │              description               │  fee   │
│ int64  │  varchar   │  varchar  │    date    │ int64 │  varchar  │                varchar                 │ varchar │ int64 │ double │  varchar   │    date    │ int64 │                varchar                 │ double │
├────────┼────────────┼───────────┼────────────┼───────┼───────────┼────────────────────────────────────────┼─────────┼───────┼────────┼────────────┼────────────┼───────┼────────────────────────────────────────┼────────┤
│      1 │ ACCT000001 │ LAB100081 │ 2024-01-01 │     1 │ LAB100081 │ Presumptive Drug Screen                │ 0309  

The join query returned the number of rows expected and did not unexpectedly multiply or omit charge records.

### Account-Level Analysis Dataset

The following query creates the account-level analysis dataset by combining charge activity with the current contract and proposed fee schedule amounts. A limited preview is displayed for verification; subsequent queries summarize the results by charge code and across the full population.

In [18]:
duckdb.sql("""
CREATE OR REPLACE TEMP VIEW account_level_analysis AS

SELECT
   v.acctNum AS "Account #",
   c.chargeNum AS "Charge #",
   m.description AS "Charge Description",
   c.qty AS "Qty",
   CAST((c.qty * m.price) AS DECIMAL(10, 2)) AS "Charge Amt",
   CAST(((c.qty * m.price) * .8) AS DECIMAL(10, 2)) AS "Current Contract Allowed Amt",
   CAST((c.qty * f.fee) AS DECIMAL(10, 2)) AS "Proposed Fee Schedule Allowed Amt",
   (CAST(((c.qty * m.price) * .8) AS DECIMAL(10, 2))) -  (CAST((c.qty * f.fee) AS DECIMAL(10, 2))) AS "Allowed Amt Difference (Current - Proposed)",
   CAST(((c.qty * m.price) - ((c.qty * m.price) * .8)) AS DECIMAL(10,2)) AS "Current Contractual Amt",
   CAST(((c.qty * m.price) - (c.qty * f.fee)) AS DECIMAL(10, 2)) AS "Proposed Fee Schedule Contractual Amt",
   (CAST(((c.qty * m.price) - ((c.qty * m.price) * .8)) AS DECIMAL(10,2)) - CAST(((c.qty * m.price) - (c.qty * f.fee)) AS DECIMAL(10, 2))) AS "Contractual Amt Difference (Current - Proposed)" 
FROM 'Lab Contract Review/patient_charges.csv' as c
JOIN 'Lab Contract Review/lab_charge_master.csv' as m
   ON c.chargeNum = m.chargeNum
JOIN 'Lab Contract Review/patient_encounters.csv' as v
   ON c.acctNum = v.acctNum
JOIN 'Lab Contract Review/lab_fee_schedule.csv' as f
   ON m.CPT = f.CPT
""")

This query displays a sample of the dataset we just created.

In [19]:
duckdb.sql("""
SELECT *
FROM account_level_analysis
ORDER BY "Account #", "Charge #"
LIMIT 20;
""")

┌────────────┬───────────┬────────────────────────────────────────┬───────┬───────────────┬──────────────────────────────┬───────────────────────────────────┬─────────────────────────────────────────────┬─────────────────────────┬───────────────────────────────────────┬─────────────────────────────────────────────────┐
│ Account #  │ Charge #  │           Charge Description           │  Qty  │  Charge Amt   │ Current Contract Allowed Amt │ Proposed Fee Schedule Allowed Amt │ Allowed Amt Difference (Current - Proposed) │ Current Contractual Amt │ Proposed Fee Schedule Contractual Amt │ Contractual Amt Difference (Current - Proposed) │
│  varchar   │  varchar  │                varchar                 │ int64 │ decimal(10,2) │        decimal(10,2)         │           decimal(10,2)           │                decimal(11,2)                │      decimal(10,2)      │             decimal(10,2)             │                  decimal(11,2)                  │
├────────────┼───────────┼───────────

## Charge Code Summary

The following query summarizes financial activity by charge code and description. It combines all patient charge records for the same laboratory test and compares the current contract allowance with the modeled allowance under the proposed fee schedule.

The allowed amount difference and contractual adjustment difference are calculated as **current minus proposed**. A positive allowed amount difference indicates that the current contract allows more than the proposed fee schedule. A positive contractual adjustment difference indicates that the proposed fee schedule would reduce the contractual write-off.

In [20]:
duckdb.sql("""
SELECT
   c.chargeNum as "Charge #",
   m.description as "Charge Description",
   SUM(c.qty) as "Net Quantity",
   SUM(CAST((c.qty * m.price) AS DECIMAL(10, 2))) AS "Total Charges",
   SUM(CAST(((c.qty * m.price) * .8) AS DECIMAL(10, 2))) AS "Total Current Contract Allowed Amt",
   SUM(CAST((c.qty * f.fee) AS DECIMAL(10, 2))) AS "Total Proposed Fee Schedule Allowed Amt",
   (SUM(CAST(((c.qty * m.price) * .8) AS DECIMAL(10, 2))) - SUM(CAST((c.qty * f.fee) AS DECIMAL(10, 2)))) AS "Total Allowed Amt Difference (Current - Proposed)",
   SUM(CAST(((c.qty * m.price) - ((c.qty * m.price) * .8)) AS DECIMAL(10,2))) AS "Total Current Contractual Amt",
   SUM(CAST(((c.qty * m.price) - (c.qty * f.fee)) AS DECIMAL(10, 2))) AS "Total Proposed Fee Schedule Contractual Amt",
   SUM((CAST(((c.qty * m.price) - ((c.qty * m.price) * .8)) AS DECIMAL(10,2)) - CAST(((c.qty * m.price) - (c.qty * f.fee)) AS DECIMAL(10, 2)))) AS "Total Contractual Amt Difference (Current - Proposed)" 
FROM 'Lab Contract Review/patient_charges.csv' as c
JOIN 'Lab Contract Review/lab_charge_master.csv' as m
   ON c.chargeNum = m.chargeNum
JOIN 'Lab Contract Review/patient_encounters.csv' as v
   ON c.acctNum = v.acctNum
JOIN 'Lab Contract Review/lab_fee_schedule.csv' as f
   ON m.CPT = f.CPT
GROUP BY c.chargeNum, m.description
ORDER BY "Total Allowed Amt Difference (Current - Proposed)" DESC; 
""")

┌───────────┬─────────────────────────────────────────────┬──────────────┬───────────────┬────────────────────────────────────┬─────────────────────────────────────────┬───────────────────────────────────────────────────┬───────────────────────────────┬─────────────────────────────────────────────┬───────────────────────────────────────────────────────┐
│ Charge #  │             Charge Description              │ Net Quantity │ Total Charges │ Total Current Contract Allowed Amt │ Total Proposed Fee Schedule Allowed Amt │ Total Allowed Amt Difference (Current - Proposed) │ Total Current Contractual Amt │ Total Proposed Fee Schedule Contractual Amt │ Total Contractual Amt Difference (Current - Proposed) │
│  varchar  │                   varchar                   │    int128    │ decimal(38,2) │           decimal(38,2)            │              decimal(38,2)              │                   decimal(38,2)                   │         decimal(38,2)         │                decimal(38,2)      

## Overall Financial Summary

The following query aggregates all laboratory charges into a single financial summary. It compares total charges, current contract allowances, proposed fee schedule allowances, and the resulting contractual adjustments across the complete analysis population.

The summary provides the overall modeled financial impact of replacing the current contract terms with the proposed fee schedule. Difference columns are calculated as **current minus proposed** and reconcile to equal amounts with opposite signs because the original charge amounts remain unchanged.

In [21]:
duckdb.sql("""
WITH line_amounts AS (
    SELECT
        CAST(c.qty * m.price AS DECIMAL(18, 2))
            AS charge_amount,

        CAST((c.qty * m.price) * 0.80 AS DECIMAL(18, 2))
            AS current_allowed_amount,

        CAST(c.qty * f.fee AS DECIMAL(18, 2))
            AS proposed_allowed_amount

    FROM 'Lab Contract Review/patient_charges.csv' AS c

    JOIN 'Lab Contract Review/lab_charge_master.csv' AS m
        ON c.chargeNum = m.chargeNum

    JOIN 'Lab Contract Review/patient_encounters.csv' AS v
        ON c.acctNum = v.acctNum

    JOIN 'Lab Contract Review/lab_fee_schedule.csv' AS f
        ON m.CPT = f.CPT
),

totals AS (
    SELECT
        SUM(charge_amount) AS total_charges,
        SUM(current_allowed_amount) AS current_allowed,
        SUM(proposed_allowed_amount) AS proposed_allowed
    FROM line_amounts
)

SELECT
    total_charges
        AS "Total Charges",

    current_allowed
        AS "Total Current Contract Allowed Amt",

    proposed_allowed
        AS "Total Proposed Fee Schedule Allowed Amt",

    current_allowed - proposed_allowed
        AS "Total Allowed Amt Difference (Current - Proposed)",

    total_charges - current_allowed
        AS "Total Current Contractual Amt",

    total_charges - proposed_allowed
        AS "Total Proposed Fee Schedule Contractual Amt",

    (total_charges - current_allowed)
        - (total_charges - proposed_allowed)
        AS "Total Contractual Amt Difference (Current - Proposed)",

    (current_allowed - proposed_allowed)
        + (
            (total_charges - current_allowed)
            - (total_charges - proposed_allowed)
          )
        AS "Reconciliation Check"

FROM totals;
""")

┌───────────────┬────────────────────────────────────┬─────────────────────────────────────────┬───────────────────────────────────────────────────┬───────────────────────────────┬─────────────────────────────────────────────┬───────────────────────────────────────────────────────┬──────────────────────┐
│ Total Charges │ Total Current Contract Allowed Amt │ Total Proposed Fee Schedule Allowed Amt │ Total Allowed Amt Difference (Current - Proposed) │ Total Current Contractual Amt │ Total Proposed Fee Schedule Contractual Amt │ Total Contractual Amt Difference (Current - Proposed) │ Reconciliation Check │
│ decimal(38,2) │           decimal(38,2)            │              decimal(38,2)              │                   decimal(38,2)                   │         decimal(38,2)         │                decimal(38,2)                │                     decimal(38,2)                     │    decimal(38,2)     │
├───────────────┼────────────────────────────────────┼────────────────────────────

## Findings and Limitations

### Findings

- Source validation confirmed the expected record counts, unique row identifiers, complete primary keys, and valid relationships between tables.
- Cross-file validation found no unexpected unmatched foreign keys or row multiplication during the joins.
- Charge corrections were accounted for using net quantity so that reversed charges did not inflate charge volume or modeled allowances.
- Under the current contract, the analyzed laboratory charges produced a total allowed amount of **\$158,093.60**.
- Applying the proposed fee schedule to the same charge activity produced a modeled allowed amount of **\$24,454.86**.
- The proposed fee schedule resulted in an allowed amount difference of **\$133,638.74**, calculated as current allowance minus proposed allowance.
- The modeled contractual adjustment changed by **-\$133,638.74**. This amount reconciles to the allowance difference with the opposite sign because the original charge amounts remain unchanged.
- The largest modeled differences were associated with **Comprehensive Metabolic Panel, Basic Metabolic Panel and Complete Blood Count with Differential**, driven by their charge volume, fee-schedule variance, or a combination of both.
- No laboratory test had a higher modeled allowance under the proposed fee schedule than under the current contract. 

### Limitations

- The project uses synthetic data and does not contain real patient, encounter, charge, or reimbursement information.
- The datasets were generated without intentional data-quality anomalies. Therefore, this project emphasizes source validation, relational integrity, and financial modeling rather than data cleaning.
- The analysis assumes that the proposed fee schedule applies uniformly to all included outpatient laboratory charges.
- Modeled allowances do not necessarily represent actual collected revenue. The analysis does not account for claim denials, patient responsibility, coordination of benefits, collection rates, or other adjudication outcomes.
- The model does not include contractual rules such as modifiers, multiple-procedure reductions, bundled services, coverage limitations, or payer-specific exceptions unless those rules are explicitly represented in the fee schedule.
- The analysis assumes that historical charge volume and test utilization would remain unchanged under the proposed fee schedule.
- Results apply only to the outpatient laboratory services represented in the synthetic dataset and should not be generalized to other departments, patient populations, or reimbursement arrangements.

## Tableau Data Export

The validated account-level analysis dataset is exported for use as the Tableau data source. The complete dataset is exported, while the notebook displays only a limited preview.

In [22]:
duckdb.sql("""
COPY (
    SELECT *
    FROM account_level_analysis
)
TO 'Lab Contract Review/lab_contract_analysis_detail.csv'
(FORMAT CSV, HEADER);
""")

In [23]:
duckdb.sql("""
SELECT COUNT(*) AS exported_rows
FROM 'Lab Contract Review/lab_contract_analysis_detail.csv';
""")

┌───────────────┐
│ exported_rows │
│     int64     │
├───────────────┤
│          1450 │
└───────────────┘

In [24]:
duckdb.sql("""
SELECT charge 

_IncompleteInputError: incomplete input (4139045092.py, line 1)